# Step E — Pen-and-paper verification via PTDF

**Task (Assignment 1, part e):** Using only the network topology and reactances, reproduce
the line flows that PyPSA computed for the **first time step** (2015-01-01 00:00). Compute
the incidence matrix, the PTDF matrix, read the nodal imbalances from the PyPSA model at
t₀, and check that `f = PTDF · p` matches `net.lines_t.p0` at t₀.

### Method

Under the DC approximation the power flow on line $l$ is:

$$f_l = \frac{1}{x_l}(\theta_{\text{from}(l)} - \theta_{\text{to}(l)})$$

In matrix form: $\mathbf{f} = \mathbf{B}\,\mathbf{K}^\top\boldsymbol{\theta}$

The nodal balance gives $\mathbf{p} = \mathbf{K}\mathbf{f}$, which combined yields:

$$\mathbf{p} = \underbrace{\mathbf{K}\,\mathbf{B}\,\mathbf{K}^\top}_{\mathbf{L}}\,\boldsymbol{\theta}$$

Since $\mathbf{L}$ is singular, we fix Denmark as slack ($\theta_0=0$) and invert the reduced system:

$$\mathbf{f} = \underbrace{\mathbf{B}\,\mathbf{K}_r^\top\,\mathbf{L}_r^{-1}}_{\text{PTDF}}\,\mathbf{p}_r$$

### Notation

| Symbol | Meaning |
|--------|----------|
| $n$ | number of buses (4: DK, DE, SE, NO) |
| $m$ | number of lines (5) |
| $\mathbf{K}$ | incidence matrix $(n \times m)$: $K_{il}=+1$ from end, $-1$ to end |
| $\mathbf{B}$ | diagonal susceptance matrix $(m \times m)$: $B_{ll}=1/x_l$ |
| $\mathbf{L}$ | weighted Laplacian $(n \times n)$: $\mathbf{L}=\mathbf{K}\mathbf{B}\mathbf{K}^\top$ |
| $\mathbf{L}_r$ | reduced Laplacian: $\mathbf{L}$ with slack row and column removed |
| $\mathbf{p}$ | nodal injections (generation $-$ demand), excluding slack |
| PTDF | $\mathbf{B}\,\mathbf{K}_r^\top\,\mathbf{L}_r^{-1}$, dimensions $(m \times (n-1))$ |


## Imports

In [1]:
from pathlib import Path
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## Rebuild the Step D network (silent)

This block reproduces the Step D setup exactly, then optimises. After this, `net` has
the same flows and capacities as at the end of Step D.


In [2]:
# ----- Cost table (same as Step A) -----
data = {
    "capital_cost": [
        1500000/25 + 60000,
        800000/25 + 14000,
        700000/25 + 24000,
    ],
    "marginal_cost": [0.0, 0.0, 9.5 * 3.6 / 0.56 + 2.30]
}
costs = pd.DataFrame(data, index=["wind_combined", "solar", "CCGT"])

# ----- Resolve paths -----
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "STEP D - electricity_demand.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent
cf_base = PROJECT_DIR / "Other country data" / "Energy charts data"

# ----- Denmark CFs + snapshot reference -----
dataframe_dk = pd.read_csv(PROJECT_DIR / "DK_2015_merged.csv", index_col=0, sep=",", parse_dates=True)
dataframe_dk.index = pd.to_datetime(dataframe_dk.index, utc=True).tz_localize(None)
dataframe_dk = dataframe_dk.sort_index()
CF_wind  = dataframe_dk["wind_cf_Unnamed: 1"].astype(float)
CF_solar = dataframe_dk["pv_cf_Unnamed: 1"].astype(float)
snapshots = dataframe_dk.index

# ----- Multi-country demand -----
demand_all = pd.read_csv(PROJECT_DIR / "STEP D - electricity_demand.csv",
                         sep=";", index_col=0, parse_dates=True)
demand_all.index = pd.to_datetime(demand_all.index, utc=True).tz_localize(None)
demand_all = demand_all.sort_index()
demand_2015 = demand_all[demand_all.index.year == 2015].copy()
demand_dk = demand_2015["DNK"].astype(float).reindex(snapshots)
demand_de = demand_2015["DEU"].astype(float).reindex(snapshots)
demand_se = demand_2015["SWE"].astype(float).reindex(snapshots)
demand_no = demand_2015["NOR"].astype(float).reindex(snapshots)

# ----- Neighbour CFs -----
cf_de_raw = pd.read_csv(cf_base / "Germany" / "Germany_hourly_capacity_factors.csv")
cf_de_raw["timestamp"] = pd.to_datetime(cf_de_raw["timestamp"])
cf_de_raw = cf_de_raw.set_index("timestamp").sort_index()
cf_de_raw.index = cf_de_raw.index.tz_localize(None)
cf_de = cf_de_raw.resample("h").mean()
cf_de["wind_combined"] = cf_de["Wind onshore"]
cf_de["solar"] = cf_de["Solar AC"]
cf_de["CCGT"]  = cf_de["Fossil gas"]
cf_de["nuclear"] = cf_de["Nuclear"]
cf_de = cf_de[["wind_combined", "solar", "CCGT", "nuclear"]].reindex(snapshots)

cf_se = pd.read_csv(cf_base / "Sweden" / "Sweden_hourly_capacity_factors.csv")
cf_se["timestamp"] = pd.to_datetime(cf_se["timestamp"])
cf_se = cf_se.set_index("timestamp").sort_index()
cf_se.index = cf_se.index.tz_localize(None)
cf_se["wind_combined"] = cf_se["Wind onshore"]
cf_se["nuclear"] = cf_se["Nuclear"]
cf_se["hydro"]   = cf_se["Hydro water reservoir"]
cf_se = cf_se[["wind_combined", "nuclear", "hydro"]].reindex(snapshots).bfill()

cf_no = pd.read_csv(cf_base / "Norway" / "Norway_hourly_capacity_factors.csv")
cf_no["timestamp"] = pd.to_datetime(cf_no["timestamp"])
cf_no = cf_no.set_index("timestamp").sort_index()
cf_no.index = cf_no.index.tz_localize(None)
cf_no["wind_combined"] = cf_no["Wind onshore"]
cf_no["hydro"] = cf_no["Hydro water reservoir"]
cf_no = cf_no[["wind_combined", "hydro"]].reindex(snapshots)

# ----- Build the network -----
net = pypsa.Network()
net.set_snapshots(snapshots)
for name, (x, y) in {"Denmark": (10.0, 56.0), "Germany": (10.5, 51.5),
                     "Sweden": (15.0, 59.5), "Norway": (10.0, 62.0)}.items():
    net.add("Bus", name, x=x, y=y)

net.add("Load", "load_DK", bus="Denmark", p_set=demand_dk.values)
net.add("Load", "load_DE", bus="Germany", p_set=demand_de.values)
net.add("Load", "load_SE", bus="Sweden",  p_set=demand_se.values)
net.add("Load", "load_NO", bus="Norway",  p_set=demand_no.values)

net.add("Carrier", ["wind_onshore", "solar", "CCGT",
                    "hydro", "nuclear", "coal", "battery"],
        color=["blue", "yellow", "brown", "cyan", "purple", "grey", "purple"])

# Denmark — extendable
net.add("Generator", "DK_wind", bus="Denmark", carrier="wind_combined",
        capital_cost=costs.loc["wind_combined", "capital_cost"],
        marginal_cost=costs.loc["wind_combined", "marginal_cost"],
        p_max_pu=CF_wind.values, p_nom_extendable=True)
net.add("Generator", "DK_solar", bus="Denmark", carrier="solar",
        capital_cost=costs.loc["solar", "capital_cost"],
        marginal_cost=costs.loc["solar", "marginal_cost"],
        p_max_pu=CF_solar.values, p_nom_extendable=True)
net.add("Generator", "DK_CCGT", bus="Denmark", carrier="CCGT",
        capital_cost=costs.loc["CCGT", "capital_cost"],
        marginal_cost=costs.loc["CCGT", "marginal_cost"],
        efficiency=0.58, p_nom_extendable=True)

# Germany — fixed
net.add("Generator", "DE_wind", bus="Germany", carrier="wind_combined",
        p_nom=41300, marginal_cost=0, p_max_pu=cf_de["wind_combined"].values,
        p_nom_extendable=False)
net.add("Generator", "DE_solar", bus="Germany", carrier="solar",
        p_nom=37000, marginal_cost=0, p_max_pu=cf_de["solar"].values,
        p_nom_extendable=False)
net.add("Generator", "DE_CCGT", bus="Germany", carrier="CCGT",
        p_nom=28360, marginal_cost=60.0, p_nom_extendable=False)
net.add("Generator", "DE_nuclear", bus="Germany", carrier="nuclear",
        p_nom=10800, marginal_cost=10.0, p_nom_extendable=False)
net.add("Generator", "DE_coal", bus="Germany", carrier="coal",
        p_nom=21420, marginal_cost=30.0, p_nom_extendable=False)

# Sweden — fixed
net.add("Generator", "SE_hydro", bus="Sweden", carrier="hydro",
        p_nom=15920, marginal_cost=5.0, p_max_pu=cf_se["hydro"].values,
        p_nom_extendable=False)
net.add("Generator", "SE_nuclear", bus="Sweden", carrier="nuclear",
        p_nom=8900, marginal_cost=10.0, p_nom_extendable=False)
net.add("Generator", "SE_wind", bus="Sweden", carrier="wind_combined",
        p_nom=5500, marginal_cost=0, p_max_pu=cf_se["wind_combined"].values,
        p_nom_extendable=False)

# Norway — fixed
net.add("Generator", "NO_hydro", bus="Norway", carrier="hydro",
        p_nom=29900, marginal_cost=5.0, p_max_pu=cf_no["hydro"].values,
        p_nom_extendable=False)
net.add("Generator", "NO_wind", bus="Norway", carrier="wind_combined",
        p_nom=700, marginal_cost=0, p_max_pu=cf_no["wind_combined"].values,
        p_nom_extendable=False)

# Transmission lines (400 kV, x = 0.1 pu)
for bus in net.buses.index:
    net.buses.loc[bus, "v_nom"] = 380
for name, bus0, bus1, s_nom, length in [
    ("line_DK_DE", "Denmark", "Germany", 3500, 360),
    ("line_DK_SE", "Denmark", "Sweden",  1700, 520),
    ("line_DK_NO", "Denmark", "Norway",  1050, 570),
    ("line_SE_NO", "Sweden",  "Norway",  3500, 480),
    ("line_DE_SE", "Germany", "Sweden",   600, 820),
]:
    net.add("Line", name, bus0=bus0, bus1=bus1,
            s_nom=s_nom, x=0.1, r=0.01, length=length, p_nom=s_nom)

# Batteries (2024 Li-ion costs, same as Step C/D)
b_cap = 100_000 / 20 + 12_500 + (150_000 / 20) * 4   # 47,500 $/MW/year
for country, bus in [("DK", "Denmark"), ("DE", "Germany"),
                     ("SE", "Sweden"),  ("NO", "Norway")]:
    net.add("StorageUnit", f"{country}_battery", bus=bus, carrier="battery",
            capital_cost=b_cap, marginal_cost=0,
            efficiency_store=0.90**0.5, efficiency_dispatch=0.90**0.5,
            max_hours=4, cyclic_state_of_charge=True, p_nom_extendable=True)

# Optimise
net.optimize(solver_name="gurobi", solver_options={"output_flag": False})
print(f"Step D rebuilt — system cost: {net.objective/1e9:.3f} B$/y")


C:\Users\terry\AppData\Local\Temp\ipykernel_21056\1968716118.py:151: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['Denmark', 'Germany', 'Sweden', 'Norway'], dtype='str', name='name')
Index(['DK_wind', 'DE_wind', 'SE_wind', 'NO_wind'], dtype='str', name='name')
DatetimeIndex(['2015-12-31 23:00:00'], dtype='datetime64[us]', name='snapshot', freq=None)
DatetimeIndex(['2015-12-31 23:00:00'], dtype='datetime64[us]', name='snapshot', freq=None)
DatetimeIndex(['2015-12-31 23:00:00'], dtype='datetime64[us]', name='snapshot', freq=None)
DatetimeIndex(['2015-12-31 23:00:00'], dtype='datetime64[us]', name='snapshot', freq=None)
DatetimeIndex(['2015-10-25 01:00:00', '2015-12-31 23:00:00'], dtype='datetime64[us]', name='snapshot', freq=None)
DatetimeIndex

Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2773895


INFO:gurobipy:Set parameter LicenseID to value 2773895


...

Reading time = 0.84 seconds


INFO:gurobipy:Reading time = 0.84 seconds


obj: 613207 rows, 262807 columns, 1116871 nonzeros


INFO:gurobipy:obj: 613207 rows, 262807 columns, 1116871 nonzeros
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 262807 primals, 613207 duals
Objective: 2.14e+10
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Line-fix-s-lower, Line-fix-s-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, Kirchhoff-Voltage-Law, StorageUnit-energy_balance were not assigned to the network.


Step D rebuilt — system cost: 21.382 B$/y


## E1 — Incidence matrix **K**

Buses (rows): 0=DK, 1=DE, 2=SE, 3=NO
Lines (columns): 0=DK→DE, 1=DK→SE, 2=DK→NO, 3=SE→NO, 4=DE→SE


In [3]:
# ── CELL E1 ─────────────────────────────────────────────────
# Build incidence matrix K from the PyPSA network.
# K[i, l] = +1  if bus i is bus0 of line l  (power flows OUT)
# K[i, l] = -1  if bus i is bus1 of line l  (power flows IN)
# K[i, l] =  0  otherwise

bus_order  = list(net.buses.index)          # ['Denmark', 'Germany', 'Sweden', 'Norway']
line_order = list(net.lines.index)          # 5 lines

n = len(bus_order)
m = len(line_order)

K = np.zeros((n, m))
for l_idx, line in enumerate(line_order):
    i_from = bus_order.index(net.lines.loc[line, "bus0"])
    i_to   = bus_order.index(net.lines.loc[line, "bus1"])
    K[i_from, l_idx] = +1
    K[i_to,   l_idx] = -1

K_df = pd.DataFrame(K, index=bus_order, columns=line_order)
print("Incidence matrix K  (rows = buses, columns = lines):")
print(K_df.to_string())


Incidence matrix K  (rows = buses, columns = lines):
         line_DK_DE  line_DK_SE  line_DK_NO  line_SE_NO  line_DE_SE
Denmark         1.0         1.0         1.0         0.0         0.0
Germany        -1.0         0.0         0.0         0.0         1.0
Sweden          0.0        -1.0         0.0         1.0        -1.0
Norway          0.0         0.0        -1.0        -1.0         0.0


## E2 — Susceptance matrix **B** and weighted Laplacian **L**

Since all lines have $x = 0.1$ pu, the susceptance of every line is $b = 1/0.1 = 10$ pu.

The weighted Laplacian matrix is defined as:

$$\mathbf{L} = \mathbf{K} \, \mathbf{B} \, \mathbf{K}^\top$$


In [4]:
# ── CELL E2 ─────────────────────────────────────────────────
# Susceptance of each line: b_l = 1 / x_l
b_values = 1.0 / net.lines['x'].values   # all = 10  (since x = 0.1)
B = np.diag(b_values)                    # diagonal susceptance matrix (m × m)

# Weighted Laplacian L = K B K^T
L = K @ B @ K.T                          # (n × n)

print('Line susceptances b = 1/x:', b_values)
print('\nWeighted Laplacian matrix L (rows/cols = buses):')
print(pd.DataFrame(L, index=bus_order, columns=bus_order).to_string())


Line susceptances b = 1/x: [10. 10. 10. 10. 10.]

Weighted Laplacian matrix L (rows/cols = buses):
         Denmark  Germany  Sweden  Norway
Denmark     30.0    -10.0   -10.0   -10.0
Germany    -10.0     20.0   -10.0     0.0
Sweden     -10.0    -10.0    30.0   -10.0
Norway     -10.0      0.0   -10.0    20.0


## E3 — PTDF matrix

Choose **Denmark (index 0) as the slack bus** (reference, $\theta_0 = 0$).
Remove its row and column from $\mathbf{L}$ to obtain the reduced, invertible Laplacian $\mathbf{L}_r$.

$$\text{PTDF} = \mathbf{B} \, \mathbf{K}_r^\top \, \mathbf{L}_r^{-1}$$

This gives a $(m \times (n-1))$ matrix mapping nodal injections to line flows.


In [5]:
# ── CELL E3 ─────────────────────────────────────────────────
slack_idx = 0  # Denmark = reference bus

# Reduced Laplacian L_r: drop row and column of slack bus
L_r = np.delete(np.delete(L, slack_idx, axis=0), slack_idx, axis=1)  # (n-1) × (n-1)

# Reduced incidence: K^T with slack-bus column removed → (m × (n-1))
KT_r = np.delete(K.T, slack_idx, axis=1)

# PTDF = B · K_r^T · L_r^{-1}   (m × (n-1))
PTDF_r = B @ KT_r @ np.linalg.inv(L_r)

# Full PTDF: insert zero column at slack position → (m × n)
PTDF = np.insert(PTDF_r, slack_idx, 0, axis=1)

non_slack = [b for b in bus_order if b != bus_order[slack_idx]]
print('Reduced Laplacian L_r:')
print(pd.DataFrame(L_r, index=non_slack, columns=non_slack).to_string())
print('\nPTDF_r  (m × (n-1)):  rows=lines, cols=non-slack buses')
print(pd.DataFrame(PTDF_r, index=line_order, columns=non_slack).round(4).to_string())


Reduced Laplacian L_r:
         Germany  Sweden  Norway
Germany     20.0   -10.0     0.0
Sweden     -10.0    30.0   -10.0
Norway       0.0   -10.0    20.0

PTDF_r  (m × (n-1)):  rows=lines, cols=non-slack buses
            Germany  Sweden  Norway
line_DK_DE   -0.625   -0.25  -0.125
line_DK_SE   -0.250   -0.50  -0.250
line_DK_NO   -0.125   -0.25  -0.625
line_SE_NO    0.125    0.25  -0.375
line_DE_SE    0.375   -0.25  -0.125


## E4 — Nodal injections at first time step

The **net injection** at each bus at time $t=0$ is:

$$p_i(t_0) = \sum_{g \in \text{bus } i} P_g(t_0) - D_i(t_0)$$

where $P_g$ is generator dispatch and $D_i$ is demand. Storage net dispatch is also
included (positive = discharging = extra injection, negative = charging = withdrawal).


In [6]:
# ── CELL E4 ─────────────────────────────────────────────────
t0 = net.snapshots[0]
print(f"First time step: {t0}")

# Generation at t0 for each generator
gen_t0 = net.generators_t.p.loc[t0]   # Series, index = generator names

# Demand at t0 for each load
load_t0 = net.loads_t.p_set.loc[t0] if t0 in net.loads_t.p_set.index else net.loads_t.p.loc[t0]

# Storage dispatch at t0 (positive = discharging, negative = charging)
stor_t0 = (net.storage_units_t.p.loc[t0]
           if t0 in net.storage_units_t.p.index
           else pd.Series(0.0, index=net.storage_units.index))

# Aggregate generation per bus
gen_per_bus = pd.Series(0.0, index=bus_order)
for gen in net.generators.index:
    bus = net.generators.loc[gen, "bus"]
    gen_per_bus[bus] += gen_t0[gen]

# Aggregate storage net injection per bus
stor_per_bus = pd.Series(0.0, index=bus_order)
for su in net.storage_units.index:
    bus = net.storage_units.loc[su, "bus"]
    stor_per_bus[bus] += stor_t0[su]

# Aggregate demand per bus
load_per_bus = pd.Series(0.0, index=bus_order)
for load in net.loads.index:
    bus = net.loads.loc[load, "bus"]
    load_per_bus[bus] += load_t0[load]

# Net injection
p_inj = gen_per_bus + stor_per_bus - load_per_bus

summary = pd.DataFrame({
    "Generation [MW]":    gen_per_bus,
    "Storage [MW]":       stor_per_bus,
    "Demand [MW]":        load_per_bus,
    "Net injection [MW]": p_inj,
})
print("\nNodal power balance at t0:")
print(summary.round(2).to_string())
print(f"\nSum of injections (must ≈ 0): {p_inj.sum():.4f} MW")


First time step: 2015-01-01 00:00:00

Nodal power balance at t0:
         Generation [MW]  Storage [MW]  Demand [MW]  Net injection [MW]
Denmark          3193.65         31.40      3210.98               14.07
Germany         45568.44          0.00     44546.00             1022.44
Sweden          16258.31          0.00     14845.00             1413.31
Norway          12848.54        172.65     15471.00            -2449.81

Sum of injections (must ≈ 0): -0.0000 MW


## E5 — Compute line flows: **f** = PTDF · **p**

With the full PTDF matrix and the nodal injection vector, the line flows are:

$$\mathbf{f} = \text{PTDF} \cdot \mathbf{p}$$

(Positive = flow from bus0 to bus1; negative = reverse direction.)


In [7]:
# ── CELL E5 ─────────────────────────────────────────────────
p_vec = p_inj.values   # (n,) array in bus_order

f_ptdf = PTDF @ p_vec  # (m,) array of line flows

f_ptdf_series = pd.Series(f_ptdf, index=line_order)
print("Line flows from PTDF calculation [MW]:")
for line, flow in f_ptdf_series.items():
    bus0 = net.lines.loc[line, "bus0"]
    bus1 = net.lines.loc[line, "bus1"]
    direction = f"{bus0} → {bus1}" if flow >= 0 else f"{bus1} → {bus0}"
    print(f"  {line}: {flow:+.2f} MW  ({direction})")


Line flows from PTDF calculation [MW]:
  line_DK_DE: -686.12 MW  (Germany → Denmark)
  line_DK_SE: -349.81 MW  (Sweden → Denmark)
  line_DK_NO: +1050.00 MW  (Denmark → Norway)
  line_SE_NO: +1399.81 MW  (Sweden → Norway)
  line_DE_SE: +336.31 MW  (Germany → Sweden)


## E6 — Verification against PyPSA

The PTDF-derived flows should match `net.lines_t.p0` at the first time step **exactly**
(within floating-point tolerance), since PyPSA also uses the DC approximation internally.


In [ ]:
# ── CELL E6 ─────────────────────────────────────────────────
# PyPSA line flows at t0 (p0 = flow from bus0 to bus1)
f_pypsa = net.lines_t.p0.loc[t0]

comparison = pd.DataFrame({
    "PTDF flow [MW]":  f_ptdf_series.round(4),
    "PyPSA flow [MW]": f_pypsa.round(4),
    "Difference [MW]": (f_ptdf_series - f_pypsa).round(6),
})

print("== Verification: PTDF vs PyPSA at t0 ==")
print(comparison.to_string())

max_err = (f_ptdf_series - f_pypsa).abs().max()
print(f"\nMax absolute error: {max_err:.6f} MW")
if max_err < 1e-3:
    print("✓ PTDF flows match PyPSA within numerical tolerance.")
else:
    print("⚠ Discrepancy detected — check sign conventions or slack bus assignment.")


=== Verification: PTDF vs PyPSA at t0 ===
            PTDF flow [MW]  PyPSA flow [MW]  Difference [MW]
line_DK_DE       -686.1236        -686.1236              0.0
line_DK_SE       -349.8111        -349.8111              0.0
line_DK_NO       1050.0000        1050.0000              0.0
line_SE_NO       1399.8111        1399.8111             -0.0
line_DE_SE        336.3125         336.3125             -0.0

Max absolute error: 0.000000 MW
✓ PTDF flows match PyPSA within numerical tolerance.
